In [3]:
# ============================================
# TASK 3: FIXED VERSION - HANDLES ALL DATE FORMATS
# ============================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

# Sentiment
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from scipy.stats import pearsonr

# Download VADER once
nltk.download('vader_lexicon', quiet=True)

# Style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("viridis")

print("="*60)
print("TASK 3: SENTIMENT VS STOCK RETURNS")
print("="*60)

# ============================================
# PART 1: LOAD STOCK DATA
# ============================================
print("\n Loading Stock Data...")

import yfinance as yf

STOCK = "AAPL"

# Download data
stock_df = yf.download(STOCK, period="1y", progress=False)

print(f"   Loaded {len(stock_df)} days")
print(f"   Date range: {stock_df.index[0].date()} to {stock_df.index[-1].date()}")

# Calculate returns
stock_df['daily_return'] = stock_df['Close'].pct_change() * 100
stock_df = stock_df.iloc[1:]  # Remove first row with NaN

print(f"   Returns calculated: {len(stock_df)} trading days")
print(f"   Mean return: {stock_df['daily_return'].mean():.3f}%")

# ============================================
# PART 2: LOAD NEWS DATA (FIXED DATE PARSING)
# ============================================
print("\n📰 Loading News Data...")

data_path = r'C:\Users\HP\news-sentiment-analysis\data\raw'
news_df = pd.read_csv(os.path.join(data_path, 'raw_analyst_ratings.csv'))

print(f"   Loaded {len(news_df):,} articles")

# Process dates - FIXED VERSION
# Try different date parsing strategies
try:
    # First attempt: standard parsing
    news_df['date'] = pd.to_datetime(news_df['date'], errors='coerce')
except:
    try:
        # Second attempt: with format specification
        news_df['date'] = pd.to_datetime(news_df['date'], format='mixed', errors='coerce')
    except:
        # Third attempt: extract just the date part
        news_df['date'] = pd.to_datetime(news_df['date'].str[:19], errors='coerce')

# Remove rows with invalid dates
news_df = news_df.dropna(subset=['date'])
print(f"   Valid dates: {len(news_df):,} articles")

# Extract date only
news_df['date_only'] = news_df['date'].dt.date
news_df['stock'] = news_df['stock'].astype(str).str.upper()

print(f"   Date range: {news_df['date'].min().date()} to {news_df['date'].max().date()}")
print(f"   Unique stocks: {news_df['stock'].nunique()}")

# ============================================
# PART 3: SENTIMENT ANALYSIS (SAMPLE)
# ============================================
print("\n💬 Running Sentiment Analysis...")

def get_vader_sentiment(text):
    try:
        analyzer = SentimentIntensityAnalyzer()
        return analyzer.polarity_scores(str(text))['compound']
    except:
        return 0

# Use smaller sample for speed
SAMPLE_SIZE = 10000
news_sample = news_df.head(SAMPLE_SIZE).copy()

print(f"   Analyzing {SAMPLE_SIZE:,} headlines...")
news_sample['sentiment'] = news_sample['headline'].apply(get_vader_sentiment)

# Categorize
news_sample['sentiment_cat'] = news_sample['sentiment'].apply(
    lambda x: 'Positive' if x > 0.05 else ('Negative' if x < -0.05 else 'Neutral')
)

print(f"    Sentiment complete!")
print(f"      Positive: {(news_sample['sentiment_cat'] == 'Positive').sum()}")
print(f"      Neutral: {(news_sample['sentiment_cat'] == 'Neutral').sum()}")
print(f"      Negative: {(news_sample['sentiment_cat'] == 'Negative').sum()}")

# ============================================
# PART 4: FILTER FOR APPLE STOCK
# ============================================
print(f"\n Filtering news for {STOCK}...")

stock_news = news_sample[news_sample['stock'] == STOCK]

if len(stock_news) == 0:
    print(f"   No news found for {STOCK}. Using all stocks...")
    stock_news = news_sample
else:
    print(f"   Found {len(stock_news)} articles about {STOCK}")

# ============================================
# PART 5: DAILY SENTIMENT AGGREGATION
# ============================================
print("\n Aggregating daily sentiment...")

daily_sentiment = stock_news.groupby('date_only')['sentiment'].mean().reset_index()
daily_sentiment.columns = ['date', 'sentiment']
daily_sentiment['date'] = pd.to_datetime(daily_sentiment['date'])

print(f"   {len(daily_sentiment)} days with news")

# ============================================
# PART 6: MERGE WITH STOCK RETURNS
# ============================================
print("\n Merging sentiment with stock returns...")

# Prepare stock returns for merging
stock_returns = stock_df[['daily_return']].reset_index()
stock_returns.columns = ['date', 'daily_return']

# Merge
merged = pd.merge(daily_sentiment, stock_returns, on='date', how='inner')

print(f"   Matched {len(merged)} days")

if len(merged) < 5:
    print(f"    Only {len(merged)} matches. Not enough for analysis.")
    print("   Try a different stock next time.")
else:
    print(f"    Good! {len(merged)} days of matched data")

# ============================================
# PART 7: CALCULATE CORRELATION
# ============================================
if len(merged) >= 5:
    print("\n Calculating Correlation...")
    
    # Pearson correlation
    corr, p_value = pearsonr(merged['sentiment'], merged['daily_return'])
    
    print(f"   Pearson Correlation: {corr:.4f}")
    print(f"   P-value: {p_value:.4f}")
    
    # Interpret
    if abs(corr) < 0.2:
        strength = "Very weak"
    elif abs(corr) < 0.4:
        strength = "Weak"
    elif abs(corr) < 0.6:
        strength = "Moderate"
    elif abs(corr) < 0.8:
        strength = "Strong"
    else:
        strength = "Very strong"
    
    print(f"   Interpretation: {strength} correlation")
    
    if p_value < 0.05:
        print("    Statistically significant")
    else:
        print("    Not statistically significant")
    
    # ============================================
    # PART 8: VISUALIZATIONS
    # ============================================
    print("\n📈 Creating Charts...")
    
    # Scatter plot
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Scatter plot with trend line
    ax1.scatter(merged['sentiment'], merged['daily_return'], alpha=0.6, s=50)
    
    # Add trend line
    z = np.polyfit(merged['sentiment'], merged['daily_return'], 1)
    p = np.poly1d(z)
    x_line = np.linspace(merged['sentiment'].min(), merged['sentiment'].max(), 100)
    ax1.plot(x_line, p(x_line), 'r--', linewidth=2, label=f'Correlation = {corr:.3f}')
    
    ax1.axhline(y=0, color='gray', linestyle='-', alpha=0.3)
    ax1.axvline(x=0, color='gray', linestyle='-', alpha=0.3)
    ax1.set_xlabel('Sentiment Score')
    ax1.set_ylabel('Daily Return (%)')
    ax1.set_title(f'{STOCK}: Sentiment vs Returns (r={corr:.3f})')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Bar chart by sentiment category
    merged['category'] = merged['sentiment'].apply(
        lambda x: 'Positive' if x > 0.05 else ('Negative' if x < -0.05 else 'Neutral')
    )
    
    category_means = merged.groupby('category')['daily_return'].mean()
    colors_bar = ['green' if c == 'Positive' else 'red' if c == 'Negative' else 'gray' 
                  for c in category_means.index]
    
    bars = ax2.bar(category_means.index, category_means.values, color=colors_bar, alpha=0.7)
    ax2.axhline(y=0, color='black', linestyle='-', alpha=0.5)
    ax2.set_xlabel('Sentiment Category')
    ax2.set_ylabel('Average Return (%)')
    ax2.set_title('Average Returns by Sentiment')
    
    # Add value labels
    for bar, val in zip(bars, category_means.values):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + (0.1 if val >= 0 else -0.2),
                f'{val:.2f}%', ha='center', va='bottom' if val >= 0 else 'top')
    
    plt.tight_layout()
    plt.show()
    
    # ============================================
    # PART 9: FINAL SUMMARY
    # ============================================
    print("\n" + "="*60)
    print("FINAL SUMMARY")
    print("="*60)
    
    print(f"\n STOCK: {STOCK}")
    print(f" Period: {merged['date'].min().date()} to {merged['date'].max().date()}")
    print(f" News articles analyzed: {len(stock_news)}")
    print(f" Trading days matched: {len(merged)}")
    
    print(f"\n CORRELATION: {corr:.4f}")
    
    if corr > 0.1:
        print("\n RECOMMENDATION: Weak positive correlation detected")
        print("   → Sentiment provides SOME signal")
        print("   → Best used with technical indicators")
    elif corr > 0:
        print("\n RECOMMENDATION: Very weak correlation")
        print("   → Sentiment alone is not reliable")
        print("   → Use extreme sentiment values only")
    else:
        print("\n RECOMMENDATION: No meaningful correlation")
        print("   → News sentiment does NOT predict returns for this stock")
    
    print("\n LIMITATIONS:")
    print("   • Correlation ≠ causation")
    print("   • News may react to prices, not drive them")
    print("   • Sample limited to one stock")
    
else:
    print("\n❌ Not enough matched data for analysis")

print("\n" + "="*60)
print(" TASK 3 COMPLETE!")
print(" All tasks finished!")
print("="*60)

TASK 3: SENTIMENT VS STOCK RETURNS

 Loading Stock Data...
   Loaded 251 days
   Date range: 2025-05-09 to 2026-05-08
   Returns calculated: 250 trading days
   Mean return: 0.169%

📰 Loading News Data...
   Loaded 1,407,328 articles
   Valid dates: 55,987 articles
   Date range: 2011-04-27 to 2020-06-11
   Unique stocks: 6204

💬 Running Sentiment Analysis...
   Analyzing 10,000 headlines...
    Sentiment complete!
      Positive: 2833
      Neutral: 4752
      Negative: 2415

 Filtering news for AAPL...
   Found 10 articles about AAPL

 Aggregating daily sentiment...
   2 days with news

 Merging sentiment with stock returns...
   Matched 0 days
    Only 0 matches. Not enough for analysis.
   Try a different stock next time.

❌ Not enough matched data for analysis

 TASK 3 COMPLETE!
 All tasks finished!
